In [1]:
import threading
import time

def formula1(x):
    return x*4 - x*5 + x + x  

def formula2(x):
    return x + x  # 2*x

def formula3(res1, res2):
    return res1 + res2

def run_iterations(n):
    res1_list = []
    res2_list = []
    res3_list = []

    start_step1 = time.time()
    
    def compute_f1():
        for i in range(n):
            res1_list.append(formula1(i))
    
    def compute_f2():
        for i in range(n):
            res2_list.append(formula2(i))

    t1 = threading.Thread(target=compute_f1)
    t2 = threading.Thread(target=compute_f2)

    t1.start()
    t2.start()
    t1.join()
    t2.join()
    
    end_step1 = time.time()
    step1_duration = end_step1 - start_step1

    start_step3 = time.time()
    for i in range(n):
        res3_list.append(formula3(res1_list[i], res2_list[i]))
    end_step3 = time.time()
    step3_duration = end_step3 - start_step3

    return step1_duration, step3_duration

for n in [10_000, 100_000]:
    step1_time, step3_time = run_iterations(n)
    print(f"Итераций: {n}")
    print(f"  Шаг 1 и 2 (параллельно) занял: {step1_time:.4f} секунд")
    print(f"  Шаг 3 (последовательно) занял: {step3_time:.4f} секунд\n")

Итераций: 10000
  Шаг 1 и 2 (параллельно) занял: 0.0033 секунд
  Шаг 3 (последовательно) занял: 0.0010 секунд

Итераций: 100000
  Шаг 1 и 2 (параллельно) занял: 0.0210 секунд
  Шаг 3 (последовательно) занял: 0.0084 секунд



In [ ]:
import time
from multiprocessing import Process, Pipe

def compute_formula1(n, conn):
    total = 0
    for x in range(1, n + 1):
        total += x**2 - x**2 + x*4 - x*5 + x + x
    conn.send(total)
    conn.close()

def compute_formula2(n, conn):
    total = 0
    for x in range(1, n + 1):
        total += x + x
    conn.send(total)
    conn.close()

def run_parallel_computation(n):
    parent_conn1, child_conn1 = Pipe()
    parent_conn2, child_conn2 = Pipe()

    p1 = Process(target=compute_formula1, args=(n, child_conn1))
    p2 = Process(target=compute_formula2, args=(n, child_conn2))

    start_time = time.time()
    p1.start()
    p2.start()

    result1 = parent_conn1.recv()
    result2 = parent_conn2.recv()

    p1.join()
    p2.join()
    time_12 = time.time() - start_time

    start_time = time.time()
    result3 = result1 + result2
    time_3 = time.time() - start_time

    return time_12, time_3, result3

def main():
    for n in [10_000, 100_000]:
        t12, t3 = run_parallel_computation(n)
        print(f"Итераций: {n}")
        print(f"Время параллельных вычислений (формулы 1 и 2): {t12:.6f} сек")
        print(f"Время вычисления формулы 3 (сложение):       {t3:.6f} сек")

main()


--- Вычисления для 10,000 итераций ---
